# 04 - Attention Mechanism Deep Dive

**Paper reference:** "Attention Is All You Need" (Vaswani et al., 2017), Section 3.2

We'll cover:
1. Scaled Dot-Product Attention (Q, K, V)
2. Why scale by √d_k
3. Multi-Head Attention
4. Masking (for the decoder)
5. Hands-on exercises

**Prerequisites:** Notebooks 01-03

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print("Ready!")

## 1. Scaled Dot-Product Attention

### The Q, K, V Analogy

Think of a **library search**:
- **Query (Q):** What you're looking for — "books about cooking"
- **Key (K):** The label/title on each book — "Italian Recipes", "Quantum Physics", ...
- **Value (V):** The actual content of each book

The attention mechanism:
1. Compare your **Query** against all **Keys** (how relevant is each book?)
2. Get a relevance score for each book
3. Return a weighted mix of **Values** (mostly the relevant books)

### The Formula (Paper Eq. 1)

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

### Step-by-Step with Numbers

Let's work through a tiny example: **4 tokens, d_k = 3**.

Imagine the sentence: "The cat sat down"

In [ ]:
# Step-by-step attention with a small example
seq_len = 4
d_k = 3
tokens = ["The", "cat", "sat", "down"]

# In practice, Q/K/V come from linear projections of input embeddings.
# Here we use small random values for clarity.
Q = torch.tensor([[1.0, 0.0, 1.0],   # The
                   [0.0, 1.0, 0.0],   # cat
                   [1.0, 1.0, 0.0],   # sat
                   [0.0, 0.0, 1.0]])  # down

K = torch.tensor([[1.0, 1.0, 0.0],   # The
                   [0.0, 1.0, 1.0],   # cat
                   [1.0, 0.0, 1.0],   # sat
                   [1.0, 1.0, 1.0]])  # down

V = torch.tensor([[1.0, 0.0],         # The — value vectors can have different dim!
                   [0.0, 1.0],         # cat
                   [1.0, 1.0],         # sat
                   [0.5, 0.5]])        # down

print(f"Q shape: {Q.shape}  (seq_len × d_k)")
print(f"K shape: {K.shape}  (seq_len × d_k)")
print(f"V shape: {V.shape}  (seq_len × d_v)")

In [ ]:
# Step 1: Q @ K^T — raw attention scores
scores = Q @ K.T
print("Step 1: Q @ K^T (raw scores)")
print(scores)
print(f"\nShape: {scores.shape} — each row shows how much each query attends to each key")
print(f"\nExample: '{tokens[0]}' attends to '{tokens[2]}' with score {scores[0, 2].item():.1f}")

In [ ]:
# Step 2: Divide by sqrt(d_k)
scale = d_k ** 0.5
scaled_scores = scores / scale
print(f"Step 2: Divide by √d_k = √{d_k} = {scale:.2f}")
print(scaled_scores)
print(f"\nScaling prevents scores from getting too large (more on this later)")

In [ ]:
# Step 3: Softmax — convert to probabilities (each row sums to 1)
attention_weights = F.softmax(scaled_scores, dim=-1)
print("Step 3: Softmax (attention weights)")
print(attention_weights)
print(f"\nRow sums: {attention_weights.sum(dim=-1)}  (all 1.0 ✓)")
print(f"\n'{tokens[1]}' attends most to: '{tokens[np.argmax(attention_weights[1].detach().numpy())]}'")

In [ ]:
# Step 4: Multiply by V — weighted combination of values
output = attention_weights @ V
print("Step 4: Attention weights @ V (output)")
print(output)
print(f"\nShape: {output.shape}  (seq_len × d_v)")
print(f"\nEach row is a weighted mix of value vectors, weighted by attention!")

In [ ]:
# Visualize the attention weights
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attention_weights.detach().numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_xlabel('Key (attending to)')
ax.set_ylabel('Query (attending from)')
ax.set_title('Attention Weights')

for i in range(seq_len):
    for j in range(seq_len):
        val = attention_weights[i, j].item()
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color='white' if val > 0.5 else 'black', fontsize=12)

plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# Let's wrap it all in a function
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention (Paper Eq. 1)
    
    Args:
        Q: Queries  (batch, seq_len, d_k)
        K: Keys     (batch, seq_len, d_k)
        V: Values   (batch, seq_len, d_v)
        mask: Optional mask to prevent attending to certain positions
    
    Returns:
        output: Weighted combination of V
        weights: Attention weights
    """
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

# Test it
out, w = scaled_dot_product_attention(Q, K, V)
print("Output matches:", torch.allclose(out, output))

## 2. Why Scale by √d_k?

**Paper Section 3.2.1:** *"We suspect that for large values of d_k, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients."*

When d_k is large, dot products become large → softmax outputs become nearly one-hot → gradients vanish.

In [ ]:
# Demo: scaling effect
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, d_k in enumerate([4, 64, 512]):
    # Random Q and K
    Q_test = torch.randn(1, 10, d_k)
    K_test = torch.randn(1, 10, d_k)
    
    # Without scaling
    scores_raw = Q_test @ K_test.transpose(-2, -1)
    weights_raw = F.softmax(scores_raw, dim=-1)
    
    # With scaling
    scores_scaled = scores_raw / (d_k ** 0.5)
    weights_scaled = F.softmax(scores_scaled, dim=-1)
    
    ax = axes[idx]
    ax.bar(range(10), weights_raw[0, 0].detach().numpy(), alpha=0.5, label='Unscaled', color='red')
    ax.bar(range(10), weights_scaled[0, 0].detach().numpy(), alpha=0.5, label='Scaled', color='blue')
    ax.set_title(f'd_k = {d_k}')
    ax.set_xlabel('Key position')
    ax.set_ylabel('Attention weight')
    ax.legend()
    ax.set_ylim(0, 1)

plt.suptitle('Without scaling, large d_k → attention becomes one-hot (peaky)', y=1.02)
plt.tight_layout()
plt.show()

print("Without scaling at d_k=512: max weight =", weights_raw[0, 0].max().item())
print("With scaling at d_k=512:    max weight =", weights_scaled[0, 0].max().item())
print("\nScaling keeps the attention distribution smoother and gradients healthier!")

## 3. Multi-Head Attention

**Paper Section 3.2.2:** Instead of one big attention, run **h parallel attention heads** with different learned projections, then concatenate.

Why? Different heads can focus on different **types of relationships**:
- Head 1 might learn syntactic relationships (subject ↔ verb)
- Head 2 might learn semantic relationships (adjective ↔ noun)
- Head 3 might learn positional patterns (nearby words)

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$
$$\text{where head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head
        
        # Projection matrices
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]
        
        # 1. Project Q, K, V
        Q = self.W_q(Q)  # (batch, seq_len, d_model)
        K = self.W_k(K)
        V = self.W_v(V)
        
        # 2. Split into heads: (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Scaled dot-product attention for each head
        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        self.attention_weights = F.softmax(scores, dim=-1)
        context = self.attention_weights @ V
        
        # 4. Concat heads: (batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 5. Final projection
        output = self.W_o(context)
        return output

# Test it
d_model = 16
num_heads = 4
seq_len = 6
batch_size = 1

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)
output = mha(x, x, x)  # self-attention: Q=K=V=x

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {mha.attention_weights.shape}")
print(f"  = (batch={batch_size}, heads={num_heads}, seq={seq_len}, seq={seq_len})")

In [ ]:
# Visualize attention patterns per head
words = ["I", "love", "deep", "learning", "very", "much"]
weights = mha.attention_weights[0].detach().numpy()  # (num_heads, seq, seq)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for h in range(num_heads):
    ax = axes[h]
    im = ax.imshow(weights[h], cmap='Blues', vmin=0, vmax=0.5)
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(words, rotation=45, ha='right')
    ax.set_yticklabels(words)
    ax.set_title(f'Head {h+1}')

plt.suptitle('Each head learns different attention patterns', fontsize=14)
plt.tight_layout()
plt.show()
print("Notice how each head has a different pattern — this is the power of multi-head attention!")

## 4. Masking: Preventing the Decoder from Cheating

In the **decoder**, when generating token at position $t$, we can only look at positions $\leq t$ (we can't see the future!).

We achieve this with a **causal mask**: set future positions to $-\infty$ before softmax, so they get zero weight.

```
Mask for seq_len=4:
  1  0  0  0     (token 0 can only see itself)
  1  1  0  0     (token 1 can see 0, 1)
  1  1  1  0     (token 2 can see 0, 1, 2)
  1  1  1  1     (token 3 can see all)
```

In [ ]:
def create_causal_mask(seq_len):
    """Create a lower-triangular mask for causal (autoregressive) attention."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

mask = create_causal_mask(4)
print("Causal mask:")
print(mask)

# Apply it to our attention function
Q_masked = torch.randn(4, 3)
K_masked = torch.randn(4, 3)
V_masked = torch.randn(4, 2)

out_unmasked, w_unmasked = scaled_dot_product_attention(Q_masked, K_masked, V_masked)
out_masked, w_masked = scaled_dot_product_attention(Q_masked, K_masked, V_masked, mask=mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, weights, title in [(axes[0], w_unmasked, 'Without Mask (Encoder)'),
                            (axes[1], w_masked, 'With Causal Mask (Decoder)')]:
    im = ax.imshow(weights.detach().numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')
    for i in range(4):
        for j in range(4):
            val = weights[i, j].item()
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    color='white' if val > 0.5 else 'black', fontsize=11)

plt.tight_layout()
plt.show()
print("With masking: upper triangle is zero — each position can only attend to past tokens!")

## 5. Exercises

### Exercise 1: Compute Attention by Hand

Given:
- Q = [[1, 0], [0, 1]]  (2 tokens, d_k = 2)
- K = [[1, 0], [0, 1]]
- V = [[5, 0], [0, 5]]

Compute attention step by step (on paper first!), then verify with code.

In [ ]:
# Exercise 1: Verify your hand computation
Q_ex = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
K_ex = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
V_ex = torch.tensor([[5.0, 0.0], [0.0, 5.0]])

# Your hand computation:
# Step 1: Q @ K^T = ?
# Step 2: Divide by sqrt(2) = ?
# Step 3: Softmax = ?
# Step 4: Multiply by V = ?

# Verify:
out_ex, w_ex = scaled_dot_product_attention(Q_ex, K_ex, V_ex)
print("Scores (Q @ K^T):")
print(Q_ex @ K_ex.T)
print(f"\nScaled scores (÷ √2 = {2**0.5:.3f}):")
print((Q_ex @ K_ex.T) / 2**0.5)
print(f"\nAttention weights:")
print(w_ex)
print(f"\nOutput:")
print(out_ex)
print("\nQ1=[1,0] is an identity query — it attends more to K1=[1,0] (similar!)")
print("So the output is mostly V1=[5,0].")

### Exercise 2: 1 Head vs 8 Heads

Compare the attention patterns of a single head (d_k = 32) vs 8 heads (d_k = 4 each).

In [ ]:
# Exercise 2
d_model = 32
seq_len = 8
x = torch.randn(1, seq_len, d_model)

mha_1 = MultiHeadAttention(d_model, num_heads=1)
mha_8 = MultiHeadAttention(d_model, num_heads=8)

_ = mha_1(x, x, x)
_ = mha_8(x, x, x)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1 head
w1 = mha_1.attention_weights[0, 0].detach().numpy()
axes[0].imshow(w1, cmap='Blues')
axes[0].set_title('1 Head (d_k=32)\nOne pattern only')

# 8 heads averaged
w8 = mha_8.attention_weights[0].detach().numpy()
axes[1].imshow(w8.mean(axis=0), cmap='Blues')
axes[1].set_title('8 Heads (d_k=4 each)\nAveraged — richer patterns')

plt.tight_layout()
plt.show()

print("8 heads can capture more diverse relationships!")
print(f"Entropy of 1-head weights: {-(w1 * np.log(w1 + 1e-10)).sum():.3f}")
print(f"Avg entropy of 8-head weights: {np.mean([-(w8[h] * np.log(w8[h] + 1e-10)).sum() for h in range(8)]):.3f}")

### Exercise 3: Attention Without Scaling

Implement attention **without** the √d_k scaling and observe how it behaves with large d_k.

In [ ]:
# Exercise 3
def attention_no_scale(Q, K, V):
    scores = Q @ K.transpose(-2, -1)  # no dividing by sqrt(d_k)!
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

# Compare with large d_k
for d_k in [4, 64, 256]:
    Q_t = torch.randn(1, 8, d_k)
    K_t = torch.randn(1, 8, d_k)
    V_t = torch.randn(1, 8, d_k)
    
    _, w_scaled = scaled_dot_product_attention(Q_t, K_t, V_t)
    _, w_unscaled = attention_no_scale(Q_t, K_t, V_t)
    
    max_scaled = w_scaled.max().item()
    max_unscaled = w_unscaled.max().item()
    print(f"d_k={d_k:3d} | Scaled max weight: {max_scaled:.4f} | Unscaled max weight: {max_unscaled:.4f}")

print("\nWithout scaling, as d_k grows, attention becomes nearly one-hot!")
print("This means gradients vanish → training becomes very difficult.")

## Summary

| Component | Formula | Purpose |
|-----------|---------|--------|
| Q, K, V | Linear projections of input | Separate "what to look for" from "what to return" |
| Q @ K^T | Dot-product similarity | How relevant is each key to each query |
| ÷ √d_k | Scaling | Prevent softmax saturation |
| Softmax | Normalize to probabilities | Attention weights sum to 1 |
| × V | Weighted sum | Output is a mix of values |
| Multi-head | h parallel attentions | Capture different relationship types |
| Mask | Set future to -inf | Prevent decoder from seeing the future |

**Next up:** Notebook 05 — The full Transformer architecture, putting it all together!